# 05 — Data Cleaning Plan

**Goal:** Turn the findings from 01 (metadata), 02 (quality), 03 (profiling), and 04
(raw exploration) into a concrete specification for dbt staging models.

This notebook is synthesis, not new analysis. It doesn't run new profiling — it reads
the shortlists already produced and organizes them into: which tables need a `stg_`
model at all, what each one needs to do, and what order to build them in.

**Structure:**
1. Table disposition — which of the 369 raw tables get individual staging attention
2. Named transformation specs — `trade_matrix`, `commodity_prices`, the 3 areacodes dedups
3. Generic transformation spec — the remaining profiling-shortlist tables (drop dead columns)
4. Naming & typing conventions
5. Suggested build order

In [1]:
from _bootstrap import project_root
from src.audit.run_management import get_latest_run_id, list_run_ids

import json
import re
import polars as pl

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(50)

from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(True)
print("Connected")

Connected


## 1. Table disposition

Classify all 369 raw tables into three buckets, based on 02/03/04's findings:

| Bucket | Count | Definition | Staging treatment |
|---|---|---|---|
| **Skip — trivial lookups** | 45 | 1-row lookup tables (03 section 8) | No individual model; reference raw directly if ever needed, or fold into a seed/dbt macro |
| **Skip — code-range lookups** | 90 | `_areacodes`/`_itemcodes` tables flagged only for expected code-range skew (03 section 8, confirmed) | No individual model; these are fine as-is, exposed via a generic `stg_lookup_*` pattern if needed |
| **Needs staging attention** | ~90 (03's actionable shortlist) | Real findings: constant/all-null columns, wide-format tables needing unpivot, confirmed duplicates | Individual `stg_` model per table (or per domain), per the specs in sections 2-3 |

This split means **dbt work concentrates on ~90-100 tables, not 369** — most of the raw
layer is either already clean or too trivial to warrant a dedicated model.

In [2]:
# Rebuild the three buckets from audit tables directly, so this notebook doesn't depend
# on polars objects still being in memory from 02/03 — reproducible from a fresh kernel.

latest_dq_run = get_latest_run_id(conn, "audit.dq_reports", layer="raw")
latest_profiling_run = get_latest_run_id(conn, "audit.profiling_reports", layer="raw")

dq = conn.execute(
    "SELECT * FROM audit.dq_reports WHERE run_id = ?", [latest_dq_run]
).pl()
profiling = conn.execute(
    "SELECT * FROM audit.profiling_reports WHERE run_id = ?", [latest_profiling_run]
).pl()

print(f"dq_reports (latest run): {dq.height} tables")
print(f"profiling_reports (latest run): {profiling.height} tables")

dq_reports (latest run): 369 tables
profiling_reports (latest run): 369 tables


In [3]:
trivial_lookups = profiling.filter(pl.col("total_rows") <= 1)

code_range_lookups = profiling.filter(
    (pl.col("total_rows") > 1)
    & (pl.col("total_columns") <= 3)
    & (pl.col("columns_with_outliers") == 1)
    & (pl.col("constant_columns") == 0)
    & (pl.col("correlation_pairs") == 0)
)

# Confirmed via generate_dedup_models.py --check-only: these three tables match the
# code_range_lookups shape but actually have exact-duplicate rows (2.00x, 2.00x, 1.96x),
# so the "safe to skip" heuristic doesn't hold for them. Carve them out so they flow
# into needs_attention and get a real dedup model instead of being silently skipped.
known_duplicated_despite_shape = {
    "foodbalancesheets_areacodes",
    "foodbalancesheetshistoric_areacodes",
    "forestry_trade_flows_areacodes",
}
code_range_lookups = code_range_lookups.filter(
    ~pl.col("table_name").is_in(known_duplicated_despite_shape)
)

skip_tables = set(trivial_lookups["table_name"]) | set(code_range_lookups["table_name"])
needs_attention = profiling.filter(~pl.col("table_name").is_in(skip_tables))

print(f"Skip — trivial lookups: {trivial_lookups.height}")
print(f"Skip — code-range lookups: {code_range_lookups.height}")
print(f"Needs staging attention: {needs_attention.height}")
print(f"Total: {trivial_lookups.height + code_range_lookups.height + needs_attention.height} / {profiling.height}")

Skip — trivial lookups: 45
Skip — code-range lookups: 87
Needs staging attention: 237
Total: 369 / 369


In [4]:
# Print the real remaining code_range_lookups table names (post-fix, 87 tables) as a
# single space-separated string -- paste this directly into:
#   make generate-dedup-models TABLES="<paste here> --check-only"
# to sweep all 87 for hidden duplication the shape-heuristic might have missed,
# rather than typing/copying table names by hand.
print(" ".join(code_range_lookups["table_name"].to_list()))

asti_expenditures_areacodes asti_researchers_areacodes climate_change_emissions_indicators_areacodes climate_change_emissions_indicators_elements commoditybalances_non_food_2010_areacodes commoditybalances_non_food_2013_old_methodology_areacodes commoditybalances_non_food_2013_old_methodology_itemcodes commoditybalances_non_food_areacodes commoditybalances_non_food_itemcodes consumerpriceindices_areacodes cost_affordability_healthy_diet_coahd_areacodes deflators_areacodes development_assistance_to_agriculture_purposes emissions_agriculture_energy_areacodes emissions_agriculture_energy_elements emissions_crops_areacodes emissions_crops_elements emissions_crops_itemcodes emissions_drained_organic_soils_areacodes emissions_drained_organic_soils_elements emissions_land_use_fires_areacodes emissions_land_use_fires_elements emissions_land_use_forests_areacodes emissions_livestock_areacodes emissions_livestock_itemcodes emissions_pre_post_production_areacodes emissions_totals_areacodes employ

## 2. Named transformation specs

These four tables have specific, confirmed transformations from 02/04 — not generic
column-dropping. Each spec below is what the corresponding dbt staging model needs to do.

### `stg_foodbalancesheets_areacodes`, `stg_foodbalancesheetshistoric_areacodes`, `stg_forestry_trade_flows_areacodes`

**Finding (confirmed via `generate_dedup_models.py --check-only` against live DuckDB data):**
these three `_areacodes` lookup tables have real exact-duplicate rows:

| table | total_rows | distinct_rows | ratio |
|---|---|---|---|
| `foodbalancesheets_areacodes` | 426 | 213 | 2.00x |
| `foodbalancesheetshistoric_areacodes` | 434 | 217 | 2.00x |
| `forestry_trade_flows_areacodes` | 423 | 216 | 1.96x |

The base tables (`foodbalancesheets`, `foodbalancesheetshistoric`, `forestry_trade_flows`)
were checked the same way and show **no duplication** (1.00x ratio, exact row-for-row
match between `total_rows` and `distinct_rows` on their full 4.8M / 11.5M / 2.8M row
counts respectively). They are ordinary generic tables — handled via the drop-list
codegen in section 3, not here.

An earlier draft of this section had this backwards (named the base tables as the
duplicated ones and claimed the `_areacodes` tables didn't exist). That was corrected
after running the actual duplication check against the real schema and data.

**Transformation:**
```sql
SELECT DISTINCT *
FROM {{ source('raw', '<table_name>') }}
```
That's the entire fix — no normalization, no fuzzy matching needed. Add a dbt test
(`dbt_utils.unique_combination_of_columns` or similar) on `Area Code` to catch
regressions if the raw loader reintroduces duplicates in a future run.


### `stg_trade_matrix`

**Finding (03, confirmed in 04):** wide-format with 78 year/flag columns
(`Y1986`...`Y2024F`). Flag values are `A`, `E`, `I`, `X`, `NULL`. 0% of rows are
all-null across every year — no pre-filter needed.

**Transformation:** unpivot from wide to long. Target grain: one row per
(reporter, partner, item, element, year).

```sql
-- Conceptual shape (use dbt_utils.unpivot or an explicit UNPIVOT):
SELECT
    "Reporter Country Code", "Reporter Countries",
    "Partner Country Code", "Partner Countries",
    "Item Code", "Item",
    "Element Code", "Element",
    "Unit",
    year,
    value,
    flag
FROM {{ source('raw', 'trade_matrix') }}
UNPIVOT (value FOR year IN (Y1986, Y1987, ..., Y2024))  -- pseudocode; DuckDB UNPIVOT or
                                                          -- dbt_utils.unpivot handles the
                                                          -- actual column enumeration
```
Join `flag` meaning from `raw.trade_flags` in a downstream model rather than staging,
to keep this model focused on shape, not enrichment. Drop true empty-value rows *after*
unpivoting (a year with a null value), not before — pre-filtering wide rows would have
wrongly dropped legitimate data present in other year columns.

### `stg_commodity_prices`

**Finding (04):** wide-format by *commodity*, not by year — one `Date` column (monthly,
e.g. `1960M01`) plus 71 commodity price columns. The 2485 correlation pairs reflect real
economic co-movement, not redundancy. Several columns are inconsistently typed `BIGINT`
vs `DOUBLE`.

**Transformation:** unpivot from wide to long. Target grain: one row per (date, commodity).

```sql
SELECT
    "Date",
    commodity,
    CAST(price AS DOUBLE) AS price  -- explicit cast to resolve BIGINT/DOUBLE inconsistency
FROM {{ source('raw', 'commodity_prices') }}
UNPIVOT (price FOR commodity IN ("Crude oil, average ($/bbl)", ..., "Silver ($/troy oz)"))
```
Parse `Date` (`1960M01` format) into a proper date/year-month type in the same model.
Commodity names currently embed their unit in the column name (e.g. `"Crude oil, average
($/bbl)"`) — consider splitting into separate `commodity` and `unit` columns during the
unpivot rather than carrying the raw compound string forward.

## 3. Generic transformation spec — remaining shortlist tables

For every other table in `needs_attention` (not one of the 4 named above), the fix is
the same repeatable pattern: drop constant and all-null columns, keep everything else
as-is. Generate the actual per-table drop list from `report_json` here, rather than
hand-writing 80+ column lists.

In [5]:
def find_constant_columns(report_json_str: str) -> list[str]:
    report = json.loads(report_json_str)
    unique_counts = report.get("unique_counts", {})
    null_counts = report.get("null_counts", {}).get("columns", {})
    return [
        col for col, info in unique_counts.items()
        if info.get("distinct_count") == 1 and null_counts.get(col, {}).get("null_pct", 0) < 100
    ]

def find_all_null_columns(report_json_str: str) -> list[str]:
    report = json.loads(report_json_str)
    null_counts = report.get("null_counts", {}).get("columns", {})
    return [col for col, info in null_counts.items() if info.get("null_pct") == 100.0]

# Resolve the 5 named-spec tables against the real table names in needs_attention,
# rather than hardcoding assumed names. Confirmed via generate_dedup_models.py
# --check-only against live DuckDB data: the three "_areacodes" tables
# (foodbalancesheets_areacodes, foodbalancesheetshistoric_areacodes,
# forestry_trade_flows_areacodes) DO exist and have real exact-duplicate rows
# (2.00x, 2.00x, 1.96x). The base tables (foodbalancesheets, foodbalancesheetshistoric,
# forestry_trade_flows) show no duplication (1.00x, checked against their full
# 4.8M/11.5M/2.8M row counts) -- they belong in the generic drop-list codegen
# (section 3), not here.
all_needs_attention_names = sorted(needs_attention["table_name"].to_list())
print("All tables in needs_attention (verify named-spec patterns against this list):")
for t in all_needs_attention_names:
    print(" ", t)

# --- Verified against the printed list above and against live duplication checks:
# trade_matrix and commodity_prices exist verbatim and need unpivoting (section 2).
# The three "_areacodes" tables exist and are the confirmed duplicates -- they get a
# SELECT DISTINCT * dedup model via generate_dedup_models.py, so they're excluded from
# the generic drop-list here. The base tables (foodbalancesheets, foodbalancesheetshistoric,
# forestry_trade_flows) have no duplication and are NOT named-spec tables -- they fall
# through to the generic path (section 3) like any other clean table.
named_spec_tables = {
    "trade_matrix",
    "commodity_prices",
    "foodbalancesheets_areacodes",
    "foodbalancesheetshistoric_areacodes",
    "forestry_trade_flows_areacodes",
}

assert len(named_spec_tables) == 5, (
    f"Expected exactly 5 named-spec tables, got {len(named_spec_tables)}: {named_spec_tables}"
)
for t in named_spec_tables:
    assert t in all_needs_attention_names, f"'{t}' not found in needs_attention -- re-check spelling"
print("Named-spec tables verified:", sorted(named_spec_tables))

generic_tables = needs_attention.filter(~pl.col("table_name").is_in(named_spec_tables))

generic_report_json = conn.execute(
    f"""
    SELECT table_name, report_json FROM audit.profiling_reports
    WHERE run_id = ? AND table_name IN ({', '.join(f"'{t}'" for t in generic_tables['table_name'].to_list())})
    """,
    [latest_profiling_run],
).pl()

drop_list_spec = generic_report_json.with_columns(
    pl.col("report_json").map_elements(find_constant_columns, return_dtype=pl.List(pl.Utf8)).alias("drop_constant"),
    pl.col("report_json").map_elements(find_all_null_columns, return_dtype=pl.List(pl.Utf8)).alias("drop_all_null"),
).select(["table_name", "drop_constant", "drop_all_null"]).filter(
    (pl.col("drop_constant").list.len() > 0) | (pl.col("drop_all_null").list.len() > 0)
).sort("table_name")

print(f"Tables with a concrete column-drop list: {drop_list_spec.height} / {generic_tables.height}")

remainder_tables = generic_tables.filter(~pl.col("table_name").is_in(drop_list_spec["table_name"].to_list()))
print(f"Tables needing only a pass-through SELECT * (no constant/all-null columns): {remainder_tables.height}")

drop_list_spec

All tables in needs_attention (verify named-spec patterns against this list):
  asti_expenditures
  asti_expenditures_archive
  asti_expenditures_archive_areacodes
  asti_expenditures_archive_elements
  asti_expenditures_flags
  asti_expenditures_indicators
  asti_researchers
  asti_researchers_archive
  asti_researchers_archive_areacodes
  asti_researchers_archive_elements
  asti_researchers_flags
  asti_researchers_indicators
  climate_change_emissions_indicators
  commodity_indices
  commodity_prices
  commoditybalances_non_food
  commoditybalances_non_food_2010
  commoditybalances_non_food_2010_elements
  commoditybalances_non_food_2010_flags
  commoditybalances_non_food_2010_itemcodes
  commoditybalances_non_food_2013_old_methodology
  commoditybalances_non_food_2013_old_methodology_flags
  commoditybalances_non_food_flags
  consumerpriceindices
  consumerpriceindices_elements
  consumerpriceindices_flags
  consumerpriceindices_itemcodes
  cost_affordability_healthy_diet_coahd
  c

table_name,drop_constant,drop_all_null
str,list[str],list[str]
"""asti_expenditures""","[""Cost Category Code"", ""Cost Category"", … ""Institution""]","[""Note""]"
"""asti_expenditures_archive""","[""Item Code"", ""Item"", ""Flag""]",[]
"""asti_researchers""","[""Degree Code"", ""Degree"", … ""Unit""]","[""Note""]"
"""asti_researchers_archive""","[""Item Code"", ""Item"", … ""Flag""]",[]
"""climate_change_emissions_indicators""","[""Flag""]",[]
"""commoditybalances_non_food""","[""Unit""]",[]
"""commoditybalances_non_food_2010""","[""Unit""]",[]
"""commoditybalances_non_food_2013_old_methodology""","[""Unit""]",[]
"""consumerpriceindices""","[""Element"", ""Note""]",[]


### Generic staging models

Each generic table gets a `stg_<table_name>` model that drops confirmed all-null columns and passes everything else through as-is (0-row tables are skipped entirely). This is handled automatically by `generate_all_staging_models.py`, which detects all-null columns directly from the database — there's no separate spec file or template to maintain here.

## 4. Naming & typing conventions for the staging layer

- **Model naming:** `stg_<raw_table_name>` (lowercase, matches raw table name exactly)
  for a 1:1 staging model. Unpivoted models keep the same base name — no `_long` suffix
  needed since staging models are long-format by convention.
- **Column naming:** convert FAOSTAT's `"Title Case With Spaces"` raw column names to
  `snake_case` in staging (e.g. `"Area Code"` → `area_code`). Do this consistently across
  every staging model, not just the flagged ones — it's the first thing a mart-layer model
  author will expect.
- **Typing:** cast numeric-looking VARCHAR columns (seen in `trade_matrix`'s early year
  columns, e.g. `Y1986` typed VARCHAR while `Y1996` is DOUBLE) to a consistent numeric type
  explicitly — don't rely on DuckDB's inferred type carrying through correctly post-unpivot.
- **Flag/note columns:** keep `flag` columns (they carry real meaning — data quality/source
  info per FAOSTAT convention) but drop `note` columns confirmed 100% null (per the
  `drop_all_null` lists above) unless a specific table's `note` column has partial data.

## 5. Suggested build order

1. **Generic pass-through/column-drop models first** (section 3) — mechanical, low-risk,
   and validates the naming/typing conventions (section 4) across a large table count
   before tackling the harder unpivots.
2. **`stg_foodbalancesheets`, `stg_foodbalancesheetshistoric`, `stg_forestry_trade_flows`
   dedup models** (section 2) — simple, and several downstream models likely join against
   these tables, so get them right early.
3. **`stg_trade_matrix`** — the highest-row-count table (6.6M rows) and most complex
   transformation (unpivot). Build and validate this in isolation before anything
   downstream depends on it.
4. **`stg_commodity_prices`** — same unpivot pattern as `trade_matrix`, should be faster
   to build once that pattern is proven out in step 3.
5. **Mart-layer work** (out of scope for this notebook) — begins only after all `stg_`
   models above are built and tested.

### Notes — Data Cleaning Plan

- *(fill in as the dbt project is actually built)*
- Confirm `trade_flags` lookup table's actual flag-code meanings before finalizing the
  `trade_matrix` downstream join (02-04 confirmed the codes exist — `A/E/I/X` — but not
  their full definitions).
- Revisit the `code_range_lookups` skip-list (section 1) if any mart-layer model ever
  needs to filter out aggregate/regional codes specifically — that filter belongs at the
  mart layer, not by modifying these raw lookup tables.
- This plan covers `raw` → `staging` only. Mart-layer modeling (fact/dimension design,
  business logic) is a separate, later planning pass once staging is built and tested.

In [6]:
conn.close()
print("Connection closed")

Connection closed
